In [1]:
2

2

In [5]:
from experiment_pipeline.config import GRANULARITIES
# from experiment_pipeline.step1_load_data import get_project_names
import json

with open("./experiment_pipeline/llm_inference_results.json", "r", encoding="utf-8") as file:
    data = json.load(file)

PROJECT_NAMES = data[GRANULARITIES[0]].keys()


invalid_counts = 0
for granularity in data.keys():
    granularity = data[granularity]
    for project_name in granularity.keys():
        project = granularity[project_name] 
        for i in project.keys():
            predicted_label = project[i]["predicted_label"]
            if "yes" in predicted_label.lower():
                predicted_label = 1
            elif "no" in predicted_label.lower():
                predicted_label = 0
            else:
                predicted_label = None
                invalid_counts += 1

            project[i]["predicted_label"] = predicted_label
#             data[granularity][project_name][i]["predicted_label"] = 0 if "no" in 

In [ ]:
import numpy as np
import sklearn.metrics as metrics

def compute_metrics(true_labels, pred_labels):
    """Computes Precision, Recall, F1, AUC, and MCC."""
    if len(true_labels) == 0:
        return [0.0] * 5
    
    p = metrics.precision_score(true_labels, pred_labels, zero_division=0)
    r = metrics.recall_score(true_labels, pred_labels, zero_division=0)
    f1 = metrics.f1_score(true_labels, pred_labels, zero_division=0)
    
    # Handle cases where only one class is present in true_labels for ROC AUC
    try:
        auc = metrics.roc_auc_score(true_labels, pred_labels)
    except ValueError:
        auc = 0.5
        
    mcc = metrics.matthews_corrcoef(true_labels, pred_labels)
    return [p, r, f1, auc, mcc]

def generate_latex_table(data_granite, data_litem):
    granularities = ["file", "class", "method", "block"]
    
    # Dynamically find all projects from the data
    projects = sorted(list(data_granite["file"].keys()))
    
    latex_lines = []
    latex_lines.append(r"\begin{table*}[t]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\caption{Evaluation Results Across Different Granularities.}")
    latex_lines.append(r"\label{tab:evaluation_results}")
    latex_lines.append(r"\resizebox{\textwidth}{!}{% Resize table to fit within text width if necessary")
    latex_lines.append(r"\begin{tabular}{ll" + "|ccccc" * (len(granularities)) + "}")
    latex_lines.append(r"\toprule")
    
    # Header Row 1
    header1 = r"\multirow{2}{*}{Project} & \multirow{2}{*}{Technique}"
    for gran in granularities:
        header1 += f" & \\multicolumn{{5}}{{c}}{{{gran.capitalize()}}}"
    header1 += r" \\"
    latex_lines.append(header1)
    
    # Midrules for each granularity group
    midrules = r"\cmidrule(lr){3-7} \cmidrule(lr){8-12} \cmidrule(lr){13-17} \cmidrule(lr){18-22}"
    latex_lines.append(midrules)
    
    # Header Row 2
    # FIX: " & & " leaves the 'Project' and 'Technique' header columns blank on row 2, shifting 'P' to column 3
    header2 = r" & & " + " & ".join(["P & R & F1 & AUC & MCC"] * len(granularities)) + r" \\"
    latex_lines.append(header2)
    latex_lines.append(r"\midrule")
    
    # Populate data rows
    for proj in projects:
        for tech_name, data_dict in [("granite", data_granite), ("LiteM", data_litem)]:
            row_str = ""
            if tech_name == "granite":
                row_str += f"\\multirow{{2}}{{*}}{{{proj}}}"
            else:
                row_str += " " # Blank space for the spanned project column
                
            row_str += f" & {tech_name}"
            
            for gran in granularities:
                # Safely get granularity and project data
                proj_gran_data = data_dict.get(gran, {}).get(proj, {})
                
                true_labels = []
                pred_labels = []
                
                # Sort indices numerically if possible
                sorted_keys = sorted(proj_gran_data.keys(), key=lambda x: int(x) if x.isdigit() else x)
                for idx in sorted_keys:
                    true_labels.append(proj_gran_data[idx]["true_label"])
                    pred_labels.append(proj_gran_data[idx]["predicted_label"])
                
                # Compute metrics
                p, r, f1, auc, mcc = compute_metrics(true_labels, pred_labels)
                row_str += f" & {p:.2f} & {r:.2f} & {f1:.2f} & {auc:.2f} & {mcc:.2f}"
            
            row_str += r" \\"
            latex_lines.append(row_str)
            
        # Add a light line or separator between different projects
        latex_lines.append(r"\hline" if proj != projects[-1] else "")
        
    latex_lines.append(r"\bottomrule")
    latex_lines.append(r"\end{tabular}")
    latex_lines.append(r"}")
    latex_lines.append(r"\end{table*}")
    
    return "\n".join(latex_lines)

In [19]:
data_granite = data
data_litem = data

print(generate_latex_table(data_granite=data, data_litem=data))

\begin{table*}[t]
\centering
\caption{Evaluation Results Across Different Granularities.}
\label{tab:evaluation_results}
\resizebox{\textwidth}{!}{% Resize table to fit within text width if necessary
\begin{tabular}{ll|ccccc||ccccc||ccccc||ccccc|}
\toprule
\multirow{2}{*}{Project} & \multirow{2}{*}{Technique} & \multicolumn{5}{c}{File} & \multicolumn{5}{c}{Class} & \multicolumn{5}{c}{Method} & \multicolumn{5}{c}{Block} \\
\cmidrule(lr){3-7} \cmidrule(lr){8-12} \cmidrule(lr){13-17} \cmidrule(lr){18-22}
 & & P & R & F1 & AUC & MCC & P & R & F1 & AUC & MCC & P & R & F1 & AUC & MCC & P & R & F1 & AUC & MCC \\
\midrule
\multirow{2}{*}{antlr4} & granite & 0.12 & 1.00 & 0.21 & 0.50 & 0.00 & 0.15 & 1.00 & 0.26 & 0.50 & 0.00 & 0.04 & 1.00 & 0.08 & 0.50 & 0.00 & 0.02 & 1.00 & 0.05 & 0.50 & 0.00 \\
  & LiteM & 0.12 & 1.00 & 0.21 & 0.50 & 0.00 & 0.15 & 1.00 & 0.26 & 0.50 & 0.00 & 0.04 & 1.00 & 0.08 & 0.50 & 0.00 & 0.02 & 1.00 & 0.05 & 0.50 & 0.00 \\
\hline
\multirow{2}{*}{guava} & granite & 0.31 &